In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import time
import datetime
import yaml
import json

import torch
from torch.utils.data import Dataset, DataLoader, RandomSampler
from torchvision.transforms import v2
from torchvision.io import decode_image
from torchvision.models import resnet50

from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
import scripts.analysis

#####

with open("config/common.yaml", "r") as common_params:
    common = yaml.safe_load(common_params)
    # unpack params
    conditions = common["conditions"]
    class_cols = common["class_cols"]
    n_classes = common["n_classes"]
    n_class_values = common["n_class_values"]

pred_cols = [col + "_pred" for col in class_cols]
confidence_cols = [col + "_confidence" for col in class_cols]

# Analyzing Model Results

This notebook is split into two sections:

* The first section shows the results of a single model
* The second section shows the results for multiple models (The results file paths *must* exist)

### Analyzing Results for a Single Model

### Analyzing Results for Multiple Models

To view results for multiple models, make sure the corresponding file path below is activated. Comment out any model file paths that should not be included in the analysis.

In [2]:
### Results data frames --> Make sure these paths are correct

test_results_file_paths = dict(
    resnet_base224 = "results/ResNet50_BASE224_test_withPredictionsAndConfidences.csv",
    resnet_base224_enhanced = "results/ResNet50_BASE224_ENHANCED_test_withPredictionsAndConfidences.csv",
    densenet_base224 = "results/DenseNet121_BASE224_test_withPredictionsAndConfidences.csv",
    densenet_base224_enhanced = "results/DenseNet121_BASE224_ENHANCED_test_withPredictionsAndConfidences.csv",
    #
    resnet_base384 = "results/ResNet50_BASE384_test_withPredictionsAndConfidences.csv",
    resnet_base384_enhanced = "results/ResNet50_BASE384_ENHANCED_test_withPredictionsAndConfidences.csv",
    densenet_base384 = "results/DenseNet121_BASE384_test_withPredictionsAndConfidences.csv",
    densenet_base384_enhanced = "results/DenseNet121_BASE384_ENHANCED_test_withPredictionsAndConfidences.csv",
    #
    # resnet_base512 = "results/ResNet50_BASE512_test_withPredictionsAndConfidences.csv",
    # resnet_base512_enhanced = "results/ResNet50_BASE512_ENHANCED_test_withPredictionsAndConfidences.csv",
    # densenet_base512 = "results/DenseNet121_BASE512_test_withPredictionsAndConfidences.csv",
    # densenet_base512_enhanced = "results/DenseNet121_BASE512_ENHANCED_test_withPredictionsAndConfidences.csv"
)

valid_rad_results_file_paths = dict(
    resnet_base224 = "results/ResNet50_BASE224_valid_rad_withPredictionsAndConfidences.csv",
    resnet_base224_enhanced = "results/ResNet50_BASE224_ENHANCED_valid_rad_withPredictionsAndConfidences.csv",
    densenet_base224 = "results/DenseNet121_BASE224_valid_rad_withPredictionsAndConfidences.csv",
    densenet_base224_enhanced = "results/DenseNet121_BASE224_ENHANCED_valid_rad_withPredictionsAndConfidences.csv",
    #
    resnet_base384 = "results/ResNet50_BASE384_valid_rad_withPredictionsAndConfidences.csv",
    resnet_base384_enhanced = "results/ResNet50_BASE384_ENHANCED_valid_rad_withPredictionsAndConfidences.csv",
    densenet_base384 = "results/DenseNet121_BASE384_valid_rad_withPredictionsAndConfidences.csv",
    densenet_base384_enhanced = "results/DenseNet121_BASE384_ENHANCED_valid_rad_withPredictionsAndConfidences.csv",
    #
    # resnet_base512 = "results/ResNet50_BASE512_valid_rad_withPredictionsAndConfidences.csv",
    # resnet_base512_enhanced = "results/ResNet50_BASE512_ENHANCED_valid_rad_withPredictionsAndConfidences.csv",
    # densenet_base512 = "results/DenseNet121_BASE512_valid_rad_withPredictionsAndConfidences.csv",
    # densenet_base512_enhanced = "results/DenseNet121_BASE512_ENHANCED_valid_rad_withPredictionsAndConfidences.csv"
)

test_rad_results_file_paths = dict(
    resnet_base224 = "results/ResNet50_BASE224_test_rad_withPredictionsAndConfidences.csv",
    resnet_base224_enhanced = "results/ResNet50_BASE224_ENHANCED_test_rad_withPredictionsAndConfidences.csv",
    densenet_base224 = "results/DenseNet121_BASE224_test_rad_withPredictionsAndConfidences.csv",
    densenet_base224_enhanced = "results/DenseNet121_BASE224_ENHANCED_test_rad_withPredictionsAndConfidences.csv",
    #
    resnet_base384 = "results/ResNet50_BASE384_test_rad_withPredictionsAndConfidences.csv",
    resnet_base384_enhanced = "results/ResNet50_BASE384_ENHANCED_test_rad_withPredictionsAndConfidences.csv",
    densenet_base384 = "results/DenseNet121_BASE384_test_withPredictionsAndConfidences.csv",
    densenet_base384_enhanced = "results/DenseNet121_BASE384_ENHANCED_test_rad_withPredictionsAndConfidences.csv",
    #
    # resnet_base512 = "results/ResNet50_BASE512_test_rad_withPredictionsAndConfidences.csv",
    # resnet_base512_enhanced = "results/ResNet50_BASE512_ENHANCED_test_rad_withPredictionsAndConfidences.csv",
    # densenet_base512 = "results/DenseNet121_BASE512_test_rad_withPredictionsAndConfidences.csv",
    # densenet_base512_enhanced = "results/DenseNet121_BASE512_ENHANCED_test_rad_withPredictionsAndConfidences.csv"
)

In [3]:
def get_all_metrics(input_file_paths_dict, include_certain_vs_uncertain=False):
    """
    Get a dictionary containing metrics at the condition level for each df in input_file_paths_dict
    Output is structured as output[df_name][analysis_type][condition][metric]

    df_name: The name of the model being analyzed
    analysis_type: What exactly is being analyzed by the metrics
        positive_vs_negative: Metrics are based on certain positive (1) and negative (0) predictions without including uncertain (2)
        certain_vs_uncertain: Metrics are based on certain (0/1) and uncertain (2) predictions without regard to accuracy
    condition: The condition of interest
    metric: The metric of interest

    certain_vs_uncertain: The radiologist-labeled datasets don't have uncertainty - set to False for these datasets
    """
    print("These models will be analyzed:")
    output_results = {}
    for df_name,path in input_file_paths_dict.items():
        print("   ", df_name)
        output_results[df_name] = {}
        df = pd.read_csv(path)
        
        # Get metrics for positive/negative predictions (uncertain not included)
        output_results[df_name]["positive_vs_negative"] = scripts.analysis.get_metrics(df)
    
        # Get metrics for certain/uncertain predictions (without regard to accuracy)
        df[class_cols + pred_cols] = df[class_cols + pred_cols].map(lambda x: 0 if x==2 else 1)
        output_results[df_name]["certain_vs_uncertain"] = scripts.analysis.get_metrics(df)
    return output_results

In [4]:
### Get metrics for test results
test_results = get_all_metrics(test_results_file_paths, include_certain_vs_uncertain=True)
    

These models will be analyzed:
    resnet_base224
    resnet_base224_enhanced
    densenet_base224
    densenet_base224_enhanced
    resnet_base384
    resnet_base384_enhanced
    densenet_base384
    densenet_base384_enhanced


In [5]:
### Get metrics for radiologist-labeled validation results
valid_rad_results = get_all_metrics(valid_rad_results_file_paths, include_certain_vs_uncertain=False)
    

These models will be analyzed:
    resnet_base224
    resnet_base224_enhanced
    densenet_base224
    densenet_base224_enhanced
    resnet_base384
    resnet_base384_enhanced
    densenet_base384
    densenet_base384_enhanced


In [6]:
# valid_rad_results

In [7]:
### Get metrics for radiologist-labeled validation results
test_rad_results = get_all_metrics(test_rad_results_file_paths, include_certain_vs_uncertain=False)
    

These models will be analyzed:
    resnet_base224
    resnet_base224_enhanced
    densenet_base224
    densenet_base224_enhanced
    resnet_base384
    resnet_base384_enhanced
    densenet_base384
    densenet_base384_enhanced


In [8]:
results = dict(
    test_results=test_results,
    valid_rad_results=valid_rad_results,
    test_rad_results=test_rad_results
)

In [9]:
### Navigating the dictionary
for results_df_name,results_df in results.items():
    """
    At this level, choose the data frame being analyzed 
    (
    test_results, test_rad_results, valid_rad_results
    )
    """
    for model_name,model in results[results_df_name].items():
        """
        At this level choose the specific model configuration being analyzed
        (
        resnet_base224, resnet_base384,
        densenet_base224, densenet_base384,
        resnet_base224_enhanced, resnet_base384_enhanced,
        densenet_base224_enhanced, densenet_base384_enhanced
        )
        """
        for metric_category,metrics in results[results_df_name][model_name].items():
            """
            At this level, choose the specific category the metrics are applied to
            (
            positive_vs_negative: 
            certain_vs_uncertain:
            )
            """
            for condition,cond_metrics in results[results_df_name][model_name][metric_category].items():
                """
                At this level, choose the specific condition being analyzed
                (
                Cardiomegaly, Consolidation, Edema, Atelectasis, Pleural Effusion
                )
                """
                for metric,value in results[results_df_name][model_name][metric_category][condition].items():
                    """
                    At this level, choose the specific metric to view
                    (
                    TP, FP, FN, TN,
                    TPR, FPR, FNR, TNR,
                    Precision, Recall,
                    BER, F1 Score
                    )
                    """
                    pass

# Example:
print(results["test_results"]["resnet_base224"]["positive_vs_negative"]["Cardiomegaly"]["TP"])

1080.0


In [16]:
### Converting the dictionary to data frame
# results_df = pd.DataFrame.from_dict(results, orient="index")
results_list = []
for results_df_name,results_data in results.items():
    for model_name,model in results[results_df_name].items():
        for metric_category,metrics in results[results_df_name][model_name].items():
            for condition,cond_metrics in results[results_df_name][model_name][metric_category].items():
                for metric,value in results[results_df_name][model_name][metric_category][condition].items():
                    results_list.append(
                        [results_df_name, 
                         model_name,
                         metric_category,
                         condition,
                         metric,
                         value
                        ]
                    )

results_df = pd.DataFrame(results_list, columns=["results_df_name", "model_name", "metric_category", "condition", "metric", "value"])

results_df.to_csv("results/model_metrics/model_metrics.csv", index=False)
results_df.head(5)

,results_df_name,model_name,metric_category,condition,metric,value
0,test_results,resnet_base224,positive_vs_negative,Cardiomegaly,TP,1080.000000
1,test_results,resnet_base224,positive_vs_negative,Cardiomegaly,FP,4140.000000
2,test_results,resnet_base224,positive_vs_negative,Cardiomegaly,FN,3680.000000
3,test_results,resnet_base224,positive_vs_negative,Cardiomegaly,TN,30471.000000
4,test_results,resnet_base224,positive_vs_negative,Cardiomegaly,TPR,0.226891


In [17]:
results_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2880 entries, 0 to 2879
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   results_df_name  2880 non-null   object 
 1   model_name       2880 non-null   object 
 2   metric_category  2880 non-null   object 
 3   condition        2880 non-null   object 
 4   metric           2880 non-null   object 
 5   value            2653 non-null   float64
dtypes: float64(1), object(5)
memory usage: 135.1+ KB


In [18]:
with open("results/model_metrics/model_metrics.json", "w") as fh:
    json.dump(results, fh)

In [19]:
### Get stats
outputs = scripts.analysis.get_metrics(results_dfs_data, results_dfs_names)
avgModelStats_byClass, avgModelStats_byValue, stats_byClass_byValue = outputs

NameError: name 'results_dfs_data' is not defined

In [ ]:
byModel_palette = {name:color for name,color in zip(results_dfs_names,
                                                    sns.color_palette(
                                                        palette="tab10", 
                                                        n_colors=len(results_dfs_names)
                                                    )
                                                   )
                  }
print(byModel_palette)

In [ ]:
def plot_avg_metrics(df, palette=None, n_plots_x=None, n_plots_y=None, grid=None, figsize=None):
    if n_plots_y is None: n_plots_y = int(len(df.columns))
    if figsize is None: figsize=(n_plots_y*1.5, 1.5)
        #
    fig, axes = plt.subplots(1, n_plots_y, figsize=figsize)
    for i,(ax,col) in enumerate(zip(axes.flatten(), df.columns)):
        plot_df = df[[col]].sort_values(by=col, ascending=False).reset_index().rename(columns={"index":"Row"})
        if i==0:
            sns.barplot(data=plot_df, x="Row", y=col, hue="Row", palette=palette, ax=ax, order=plot_df["Row"], width=0.75, gap=0.025, legend="brief")
        else:
            sns.barplot(data=plot_df, x="Row", y=col, hue="Row", palette=palette, ax=ax, order=plot_df["Row"], width=0.75, gap=0.025)
        sns.barplot(data=plot_df, x="Row", y=col, hue="Row", palette=palette, ax=ax, order=plot_df["Row"], width=0.75, gap=0.025)
        ax.set(ylim=(0,1), ylabel="", xlabel="", xticks="")
        ax.set_title(col, fontsize=8)
        ax.tick_params(axis='y', labelsize=6)
        if grid:
            ax.grid(axis="x")
        if i==0:
            sns.move_legend(ax, loc="lower left", bbox_to_anchor=(len(df.columns)//6,-.5), ncol=len(df.columns)//2, frameon=False, fontsize=7, title="")
    plt.subplots_adjust(wspace=0.4, hspace=0.6)
    return fig,axes

In [ ]:
### Visualize
plot_fig, plot_axes = plot_avg_metrics(avgModelStats_byClass, byModel_palette)
#
print("Average Results by Model")
avgModelStats_byClass

In [ ]:
print("Breakdown of contents for stats_byClass_byValue")
for key,values in stats_byClass_byValue.items():
    print(f"Key1: {key}")
    for key2,values2 in values.items():
        print(f"\tKey2: {key2}")

In [ ]:
### Example data
# stats_byClass_byValue["densenet_base224_enhanced"]["Stats by Class by Value"]

In [ ]:
### Example data
stats_byClass_byValue["densenet_base224_enhanced"]["Avg Stats by Value"]

In [ ]:
for i,name in enumerate(results_dfs_names):
    print(f"{i}: {name}")

In [ ]:
### View stats by training image type
name_ind = 10


output_col = results_dfs_names[name_ind]
input_df = results_dfs_data[name_ind]
#
n_plots_x = 3
n_plots_y = 2
figsize = (20,16)
#
stats_dict, stats_dict_byClass = scripts.analysis.get_metrics_byClass(input_df,
                                                                      figsize=figsize,
                                                                      **metrics_kwargs)
avg_metrics_byClass = stats_dict["Avg Stats by Class"].loc[["Avg Class Precision", 
                                                            "Avg Class Recall", 
                                                            "Avg Class F1 Score",
                                                            "Avg Class TPR", 
                                                            "Avg Class TNR", 
                                                            "Avg Class BER"],:].T
color_map = {}
plot_fig, plot_axes = scripts.analysis.plot_avg_metrics(avg_metrics_byClass, 
                                                        byClass_palette, 
                                                        n_plots_x, 
                                                        n_plots_y, 
                                                        figsize=figsize, 
                                                        grid=True)
plot_fig.suptitle(f"Average Metrics for Each Condition\n({output_col})")
plot_fig.tight_layout()
avg_metrics_byClass

In [ ]:
metrics_cols = ["Avg Class Precision", 
               "Avg Class TPR",
               "Avg Class Recall", 
               "Avg Class TNR",
               "Avg Class F1 Score",
               "Avg Class BER"]